In [ ]:
# Total Up-Regulated Pathways Bar Plot
library(tidyverse)
library(readxl)

# Define function labels for clusters
cluster_functions <- tibble(
  Cluster = paste("cluster", c(1,9,7,8,14,17,19,5,2,3,15,4,20,11,13,6,10,18,16,12)),
  Function = c("Mitochondrial respiration / ATP synthesis",
               "Synaptic Transmission / Vesicle-Mediated Transport",
               "Vesicle Transport / Membrane Trafficking",
               "Phospholipid Metabolism / Endocytosis",
               "Axon guidance / Nervous system development",
               "Protein folding / Protein catabolic process",
               "Ribosome biogenesis / rRNA processing",
               "Ribosome biogenesis / Synapse organization",
               "Lipid transport / Phospholipid metabolism",
               "Vitamin transport / Sulfur compound transport",
               "Angiogenesis / Nervous system development",
               "ECM organization / MAPK cascade",
               "Cytoskeleton organization / Neuron development",
               "Cytoskeleton/actin / Cell junction/adhesion",
               "Interferon/antiviral / Cytokine signaling",
               "Immune Response / Cell Activation",
               "MHC/antigen presentation / Endosome/lysosome",
               "Immune response / Endocytosis",
               "Antigen processing and presentation / Viral infection",
               "T cell activation / Cell adhesion")
)

# Read and process data
raw <- read_excel("path/to/total-up-ai-update.xlsx", col_names = FALSE)
colnames(raw) <- c("Cluster", "EXC_AD", "EXC_AGE", "INH_AD", "INH_AGE", "OLG_AD", "OLG_AGE",
                   "OPC_AD", "OPC_AGE", "AST_AD", "AST_AGE", "EC_AD", "EC_AGE", "PER_AD", "PER_AGE",
                   "FIB_AD", "FIB_AGE", "MIC_AD", "MIC_AGE", "MAC_AD", "MAC_AGE", "TC_AD", "TC_AGE")

plot_data <- raw %>%
  filter(str_detect(Cluster, "^cluster")) %>%
  mutate(across(-Cluster, as.numeric)) %>%
  pivot_longer(-Cluster, names_to = "Group", values_to = "Value") %>%
  separate(Group, into = c("CellType", "Phenotype"), sep = "_") %>%
  inner_join(cluster_functions, by = "Cluster") %>%
  mutate(Function_Order = factor(Function, levels = rev(cluster_functions$Function)),
         Phenotype = factor(Phenotype, levels = c("AD", "AGE")),
         CellType = factor(CellType, levels = c("EXC","INH","OLG","OPC","AST","EC","PER","FIB","MIC","MAC","TC")))

# Plot
p <- ggplot(plot_data, aes(x = Value, y = Function_Order, fill = Phenotype)) +
  geom_col(width = 0.8, position = position_dodge(preserve = "single")) +
  facet_grid(~ CellType + Phenotype, scales = "free_x", space = "fixed") +
  scale_x_continuous(limits = c(0, 300), expand = c(0,0)) +
  scale_fill_manual(values = c("AD" = "#d94f5e", "AGE" = "#f08a30")) +
  labs(x = "Count", y = NULL, title = "Total Up-Regulated Pathways") +
  theme_minimal() + theme(axis.text.y = element_text(size = 8), panel.grid.major.y = element_blank())
print(p)

# Total Down-Regulated Pathways Bar Plot

# Similar structure with down-regulated data
# (code omitted for brevity, but follows the same pattern)

# EXC and INH Enrichment by Brain Region

# Example for EXC (similar for INH)
raw_exc <- read_excel("path/to/region_exc.xlsx", col_names = FALSE)
colnames(raw_exc) <- c("Cluster", "PFC_down", "PFC_up", "SFG_down", "SFG_up", "OC_down", "OC_up", "OTC_down", "OTC_up")

plot_exc <- raw_exc %>%
  filter(str_detect(Cluster, "^cluster")) %>%
  mutate(across(-Cluster, as.numeric)) %>%
  pivot_longer(-Cluster, names_to = "Group", values_to = "Value") %>%
  separate(Group, into = c("Region", "Direction"), sep = "_") %>%
  inner_join(cluster_functions, by = "Cluster") %>%
  mutate(Function_Order = factor(Function, levels = rev(cluster_functions$Function)),
         Region = factor(Region, levels = c("PFC","SFG","OC","OTC")),
         Direction = factor(Direction, levels = c("up","down")))

p <- ggplot(plot_exc, aes(x = Value, y = Function_Order, fill = Direction)) +
  geom_col(width = 0.8, position = position_dodge(preserve = "single")) +
  facet_grid(~ Region, scales = "free_x", space = "fixed") +
  scale_x_continuous(limits = c(0, 350), expand = c(0,0)) +
  scale_fill_manual(values = c("up" = "#d94f5e", "down" = "#2c7bb6")) +
  labs(x = "Counts", y = NULL, title = "EXC Enrichment by Region") +
  theme_minimal() + theme(axis.text.y = element_text(size = 8))
print(p)

In [ ]:
cat("\n========== PART 1: Pathway Enrichment Bar Plots ==========\n")

# --- USER: Update this path to your enrichment Excel file ---
enrichment_file <- "path/to/AD_age_new.xlsx"

# Read the Excel file containing enrichment results
df <- read_excel(enrichment_file)
df$Status <- toupper(df$Status)   # ensure uppercase

# Define fixed sets of pathways
up_pathways <- c(
    "inflammatory response", "Cytokine Signaling in Immune system", 
    "regulation of chemotaxis", "positive regulation of MAPK cascade", 
    "angiogenesis", "regulation of vasculature development",
    "positive regulation of endothelial cell migration", 
    "tissue morphogenesis", "wound healing"
)
down_pathways <- c(
    "GABAergic synapse", "regulation of synapse organization",
    "cell-cell junction", "cell-cell adhesion", 
    "GTPase regulator activity", "T cell receptor signaling", 
    "T cell activation", "T cell differentiation",
    "Th17 cell differentiation pathway", "Th1 and Th2 cell differentiation"
)

cell_types <- unique(df$Celltype)

# -------- Upregulated pathways ----------
up_plots <- list()
for (cell in cell_types) {
    # Extract data for this cell type and UP status
    subset_df <- df %>% 
        filter(Celltype == cell, Status == "UP") %>%
        select(Description, `Log(q-value)`) %>%
        mutate(
            Description = trimws(Description),
            `Log(q-value)` = as.numeric(`Log(q-value)`)
        ) %>%
        filter(!is.na(`Log(q-value)`), Description %in% up_pathways)
    
    # Create a full data frame with all pathways (fill missing with 0)
    full_df <- data.frame(
        Description = up_pathways,
        `Log(q-value)` = 0.0,
        stringsAsFactors = FALSE
    )
    if (nrow(subset_df) > 0) {
        temp <- data.frame(Description = up_pathways) %>%
            left_join(subset_df, by = "Description") %>%
            replace_na(list(`Log(q-value)` = 0.0))
        full_df$`Log(q-value)` <- temp$`Log(q-value)`
    }
    full_df$Neg_Log_qvalue <- -full_df$`Log(q-value)`
    full_df$Description <- factor(full_df$Description, levels = rev(up_pathways))
    
    # Plot
    p <- ggplot(full_df, aes(x = Neg_Log_qvalue, y = Description)) +
        geom_bar(stat = "identity", fill = "#3B9AB2", width = 0.3) +
        labs(
            x = "-Log(q-value)",
            y = "Pathway",
            title = paste0(cell, " - UP")
        ) +
        theme_minimal() +
        theme(
            plot.title = element_text(hjust = 0.5, size = 12, face = "bold"),
            axis.text = element_text(size = 8),
            axis.title = element_text(size = 10)
        )
    up_plots[[paste0(cell, "_UP")]] <- p
}

# Hide y-axis for all but the first plot (for side‑by‑side combination)
if (length(up_plots) > 1) {
    for (i in 2:length(up_plots)) {
        up_plots[[i]] <- up_plots[[i]] + 
            theme(
                axis.title.y = element_blank(),
                axis.text.y = element_blank(),
                axis.ticks.y = element_blank()
            )
    }
}
# Combine using patchwork
combined_up <- wrap_plots(up_plots, ncol = length(up_plots)) +
    plot_annotation(title = "UP Pathway Enrichment") &
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"))
ggsave("UP_pathway_enrichment.pdf", combined_up,
       width = 6 * length(up_plots), height = 8, dpi = 300)
cat("UP pathway enrichment plot saved.\n")

# -------- Downregulated pathways ----------
down_plots <- list()
for (cell in cell_types) {
    # Use vectorized approach to avoid joining issues
    filter_vec <- (df$Celltype == cell) & (df$Status == "DOWN") & (df$Description %in% down_pathways)
    desc_vec <- df$Description[filter_vec]
    log_q_vec <- as.double(df$`Log(q-value)`[filter_vec])
    
    result_log_q <- rep(0.0, length(down_pathways))
    if (length(desc_vec) > 0) {
        match_pos <- match(desc_vec, down_pathways)
        result_log_q[match_pos] <- log_q_vec
    }
    neg_log_q <- -result_log_q
    
    plot_df <- data.frame(
        Description = factor(down_pathways, levels = rev(down_pathways)),
        Neg_Log_qvalue = neg_log_q
    )
    
    p <- ggplot(plot_df, aes(x = Neg_Log_qvalue, y = Description)) +
        geom_bar(stat = "identity", fill = "#9BD1E5", width = 0.3) +
        labs(
            x = "-Log(q-value)",
            y = "Pathway",
            title = paste0(cell, " - DOWN")
        ) +
        theme_minimal() +
        theme(
            plot.title = element_text(hjust = 0.5, size = 12, face = "bold"),
            axis.text = element_text(size = 8),
            axis.title = element_text(size = 10)
        )
    down_plots[[paste0(cell, "_DOWN")]] <- p
}

if (length(down_plots) > 1) {
    for (i in 2:length(down_plots)) {
        down_plots[[i]] <- down_plots[[i]] + 
            theme(
                axis.title.y = element_blank(),
                axis.text.y = element_blank(),
                axis.ticks.y = element_blank()
            )
    }
}
combined_down <- wrap_plots(down_plots, ncol = length(down_plots)) +
    plot_annotation(title = "DOWN Pathway Enrichment") &
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"))
ggsave("DOWN_pathway_enrichment.pdf", combined_down,
       width = 6 * length(down_plots), height = 8, dpi = 300)
cat("DOWN pathway enrichment plot saved.\n")

# ============================================================================